In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import spacy
import re
import contractions
from textblob import TextBlob
import os
import requests


# 1. LOAD THE DOCUMENT

In [30]:
data=open('data3.txt').read()
data

"Machine Learning is a technique that allows computers to learn from data and make decisions without explicit programming. It works by identifying patterns in data and using them to make predictions. It is used in areas such as:\n\nImage Recognition\nSpeech Processing\nLanguage Translation\nRecommender Systems\n213124124.webp213124124.webp\nNeed for Machine Learning\nMachine Learning is important because traditional programming cannot handle complex tasks or large amounts of data efficiently. ML overcomes this by learning from data and making predictions without fixed rules. It is needed for the following reasons:\n\n1. Solving Complex Business Problems\nTraditional programming struggles with tasks like language understanding and medical diagnosis. ML learns from data and predicts outcomes easily.\n\nExamples:\n\nImage and speech recognition in healthcare.\nLanguage translation and sentiment analysis.\n2. Handling Large Volumes of Data\nThe internet generates huge amounts of data every

# 2. TEXT NORMALIZATION

### converting all the characters into lower case

In [31]:
data=data.lower()
data

"machine learning is a technique that allows computers to learn from data and make decisions without explicit programming. it works by identifying patterns in data and using them to make predictions. it is used in areas such as:\n\nimage recognition\nspeech processing\nlanguage translation\nrecommender systems\n213124124.webp213124124.webp\nneed for machine learning\nmachine learning is important because traditional programming cannot handle complex tasks or large amounts of data efficiently. ml overcomes this by learning from data and making predictions without fixed rules. it is needed for the following reasons:\n\n1. solving complex business problems\ntraditional programming struggles with tasks like language understanding and medical diagnosis. ml learns from data and predicts outcomes easily.\n\nexamples:\n\nimage and speech recognition in healthcare.\nlanguage translation and sentiment analysis.\n2. handling large volumes of data\nthe internet generates huge amounts of data every

# REMOVING EXTRA SPACES

In [32]:
data= re.sub(r'\s{2,}','',data)
data

"machine learning is a technique that allows computers to learn from data and make decisions without explicit programming. it works by identifying patterns in data and using them to make predictions. it is used in areas such as:image recognition\nspeech processing\nlanguage translation\nrecommender systems\n213124124.webp213124124.webp\nneed for machine learning\nmachine learning is important because traditional programming cannot handle complex tasks or large amounts of data efficiently. ml overcomes this by learning from data and making predictions without fixed rules. it is needed for the following reasons:1. solving complex business problems\ntraditional programming struggles with tasks like language understanding and medical diagnosis. ml learns from data and predicts outcomes easily.examples:image and speech recognition in healthcare.\nlanguage translation and sentiment analysis.\n2. handling large volumes of data\nthe internet generates huge amounts of data every day. machine le

# REMOVING NUMBERS LIKE 1.2,...

In [33]:
data=re.sub(r'\d+\.','',data)
data

"machine learning is a technique that allows computers to learn from data and make decisions without explicit programming. it works by identifying patterns in data and using them to make predictions. it is used in areas such as:image recognition\nspeech processing\nlanguage translation\nrecommender systems\nwebpwebp\nneed for machine learning\nmachine learning is important because traditional programming cannot handle complex tasks or large amounts of data efficiently. ml overcomes this by learning from data and making predictions without fixed rules. it is needed for the following reasons: solving complex business problems\ntraditional programming struggles with tasks like language understanding and medical diagnosis. ml learns from data and predicts outcomes easily.examples:image and speech recognition in healthcare.\nlanguage translation and sentiment analysis.\n handling large volumes of data\nthe internet generates huge amounts of data every day. machine learning processes and ana

# CONTRACTIONS

In [34]:
data = contractions.fix(data)
data

'machine learning is a technique that allows computers to learn from data and make decisions without explicit programming. it works by identifying patterns in data and using them to make predictions. it is used in areas such as:image recognition\nspeech processing\nlanguage translation\nrecommender systems\nwebpwebp\nneed for machine learning\nmachine learning is important because traditional programming cannot handle complex tasks or large amounts of data efficiently. ml overcomes this by learning from data and making predictions without fixed rules. it is needed for the following reasons: solving complex business problems\ntraditional programming struggles with tasks like language understanding and medical diagnosis. ml learns from data and predicts outcomes easily.examples:image and speech recognition in healthcare.\nlanguage translation and sentiment analysis.\n handling large volumes of data\nthe internet generates huge amounts of data every day. machine learning processes and ana

# REMOVING PUNCTUATIONS AND SPECIAL CHARACTERS

In [35]:
data= re.sub(r'[^0-9a-zA-Z\s]','',data)

# TEXTBLOB

In [36]:
values = TextBlob(data).correct()
values

TextBlob("machine learning is a technique that allows computers to learn from data and make decisions without explicit programming it works by identifying patterns in data and using them to make predictions it is used in areas such image recognition
speech processing
language translation
recommended systems
webpwebp
need for machine learning
machine learning is important because traditional programming cannot handle complex tasks or large amounts of data efficiently my overcome this by learning from data and making predictions without fixed rules it is needed for the following reasons solving complex business problems
traditional programming struggles with tasks like language understanding and medical diagnosis my learns from data and predict outcome easilyexamplesimage and speech recognition in healthcare
language translation and sentiment analysis
 handling large volumes of data
the internet generate huge amounts of data every day machine learning processes and analyze this data quic

# LEMMATIZATION

In [37]:
nlp=spacy.load('en_core_web_sm')
tokens=nlp(data)
updated_tokens= [token.lemma_ for token in tokens if not token.is_stop]
data=' '.join(updated_tokens)
data

'machine learning technique allow computer learn datum decision explicit programming work identify pattern datum prediction area asimage recognition \n speech processing \n language translation \n recommender system \n webpwebp \n need machine learn \n machine learning important traditional programming handle complex task large amount datum efficiently ml overcome learn datum make prediction fix rule need following reason solve complex business problem \n traditional programming struggle task like language understanding medical diagnosis ml learn datum predict outcome easilyexamplesimage speech recognition healthcare \n language translation sentiment analysis \n  handle large volume datum \n internet generate huge amount datum day machine learning process analyze datum quickly provide valuable insight real time predictionsexamplesfraud detection financial transaction \n personalize feed recommendation facebook instagram billion interaction \n  automate repetitive task \n ml automate ti

# CHUNKING(converting dox--> chunks)

In [38]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)

chunks=list(set(splitter.split_text(data)))

# EMBEDDINGS

In [39]:
embedding_model= SentenceTransformer(
    model_name_or_path='sentence-transformers/all-miniLM-L6-V2'
)
chunk_embeddings=embedding_model.encode(chunks).astype('float32')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [40]:
dimension=chunk_embeddings.shape[1]
dimension

384

In [41]:
faiss.normalize_L2(chunk_embeddings)

In [42]:
index_faiss_db= faiss.IndexFlatIP(dimension)
index_faiss_db.add(chunk_embeddings)

In [ ]:
def rag_query(query, k=2,): 
    #faiss.normalize_L2(query)
    query_embeddings=embedding_model.encode(query).astype('float32')
    query_embeddings=query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance, index= index_faiss_db.search(query_embeddings, k=k)
    R_chunks=[chunks[i] for i in index[0]]
    r_str=' '.join(R_chunks)
    
    prompt = f'''
            you are a helpful assistant
            assigned task for you : structure my output => {r_str}
            note :  
            1)Don't add extra contents just structured mentioned output.
            2)if there is mistake in output please correct or else please keep the original output
            with structure result.
            '''
    import os
import requests

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

response = query({
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "model": "deepseek-ai/DeepSeek-R1:novita"
})

print(response)
    

user_prompt= 'Explain machine learning?'
user_prompt=re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

response=rag_query(user_prompt)
print(response)

NameError: name 'prompt' is not defined

In [54]:
def g_text(query, k=2,): 
    #faiss.normalize_L2(query)
    query_embeddings=embedding_model.encode(query).astype('float32')
    query_embeddings=query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance, index= index_faiss_db.search(query_embeddings, k=k)
    R_chunks=[chunks[i] for i in index[0]]
    R_str =' '.join(R_chunks)
prompt=f'''
            you are a helpful assistant
            assigned task for you : structure my output => {R_str}
            note :  
            1)Don't add extra contents just structured mentioned output.
            2)if there is mistake in output please correct or else please keep the original output
            with structure result.
            '''
import os
import requests

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

response = query({
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "model": "deepseek-ai/DeepSeek-R1:novita"
})
    
    
return response
user_prompt= 'Explain machine learning?'
user_prompt=re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

response=rag_query(user_prompt)
print(response)

NameError: name 'R_str' is not defined